# Exercise 11: Cross-Document Synthesis

Test questions that require **combining information from multiple chunks**.

**Goal:** Assess if RAG can synthesize coherent answers from scattered evidence.


In [1]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [2]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [3]:
from pathlib import Path

CORPUS_SUBFOLDER = "ModelTService"  # <- change if needed
DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = f"{DRIVE_BASE}/{CORPUS_SUBFOLDER}"
else:
    DOC_FOLDER = str(Path("Corpora") / CORPUS_SUBFOLDER)

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora/ModelTService


In [4]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)

Loaded 16 documents


In [5]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks

Total chunks: 1286


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


Loading: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [7]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "modelTService_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


✓ Cache loaded: 1286 chunks, 1286 vectors


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded ✓


In [9]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [10]:
import time

SYNTHESIS_QUERIES = [
    "What are ALL the maintenance tasks I need to do on the Model T engine?",
    "Compare the procedures for adjusting the carburetor vs. adjusting the transmission bands.",
    "What tools do I need for a complete Model T tune-up?",
    "Summarize all safety warnings mentioned in the manual.",
    "What are all the lubrication points on a Model T and what lubricant does each need?",
]

K_VALUES = [3, 5, 10]

for q in SYNTHESIS_QUERIES:
    print(f"\n{'='*70}")
    print(f"SYNTHESIS QUERY: {q}")
    print(f"{'='*70}")
    for k in K_VALUES:
        print(f"\n  [k={k}]")
        t0 = time.time()
        results = retrieve(q, top_k=k)
        # Show source diversity
        sources = list(set(c.source_file for c, _ in results))
        print(f"  Sources ({len(sources)} unique): {sources}")
        answer = rag_query(q, top_k=k)
        print(f"  Answer ({time.time()-t0:.1f}s): {answer[:400]}")



SYNTHESIS QUERY: What are ALL the maintenance tasks I need to do on the Model T engine?

  [k=3]
  Sources (2 unique): ['Ford-Model-T-Man-1919-ocr.pdf', 'ModelT-01-10-ocr.pdf']
  Answer (13.5s): To keep the Model T engine operating at its best and within budget, here are some key maintenance tasks:

1. **Regular Oil Changes**: Ensure the engine receives fresh oil regularly to maintain lubrication and prevent wear.

2. **Water Checks**: Keep an eye on coolant levels as well, especially if the vehicle is exposed to hot weather or dusty conditions.

3. **Oil Filter Replacement**: Periodicall

  [k=5]
  Sources (3 unique): ['Ford-Model-T-Man-1919-ocr.pdf', 'Ford-Model-T-Man-1919.txt', 'ModelT-01-10-ocr.pdf']
  Answer (11.1s): The entire Ford organization is interested in maintaining the constant operation of each individual Ford car at the lowest possible cost. They know that many cars have been poorly maintained by unskilled repairmen. A new machine requires more careful attention while 

In [11]:
# Check: does the model miss info that wasn't in top-k?
probe_q = "What are ALL the maintenance tasks I need to do on the Model T engine?"

print("Retrieving top 20 chunks to see what exists in corpus:")
results_20 = retrieve(probe_q, top_k=20)
for i, (chunk, score) in enumerate(results_20, 1):
    print(f"  [{i:2d}] score={score:.3f} | {chunk.source_file} | {chunk.text[:100]}")

print("\nNow generating answer with k=3 vs k=20:")
for k in [3, 20]:
    print(f"\n[k={k}]")
    print(rag_query(probe_q, top_k=k)[:500])


Retrieving top 20 chunks to see what exists in corpus:
  [ 1] score=0.451 | ModelT-01-10-ocr.pdf | .
entire Ford orgenization is interested in keeping every individual Ford. car
constant operation, a
  [ 2] score=0.451 | Ford-Model-T-Man-1919-ocr.pdf | .
entire Ford orgenization is interested in keeping every individual Ford. car
constant operation, a
  [ 3] score=0.431 | ModelT-01-10-ocr.pdf | ll the running speeds needed! for drdinery.
Ce ee eee ee ee tine ce wae erties Daye eae
File the cut
  [ 4] score=0.431 | Ford-Model-T-Man-1919-ocr.pdf | ll the running speeds needed! for drdinery.
Ce ee eee ee ee tine ce wae erties Daye eae
File the cut
  [ 5] score=0.427 | Ford-Model-T-Man-1919.txt | far.
6. Gas mixture too rich.

7. Water circulation retarded ‘by
sediment in radiator. .

8 Dirty sp
  [ 6] score=0.425 | ModelT-61-62.txt | What It Is For
How to Disconnect

OPERATION
Alustments—What to Do vere... cong

Model T Truck

‘Worm
  [ 7] score=0.422 | ModelT-61-62-ocr.pdf | [Page 1]
Wha

## Analysis

| Query | k=3 complete? | k=5 complete? | k=10 complete? | Missing info? |
|-------|--------------|--------------|----------------|---------------|
| All maintenance tasks | ⚠️ Partial (hallucinated oil filter) | ❌ Too meta (quoted Ford org statement, no tasks) | ✅ Best — 6 concrete tasks | k=3/5 miss running gear inspection, spark plug checks |
| Carb vs. transmission comparison | ⚠️ Confused (said foot pedal = transmission adj.) | ⚠️ Vague (dash knob only, no detail) | ⚠️ Slightly better, still simplified | All k values miss specific step-by-step comparison |
| Tune-up tool list | ⚠️ Basic (4 tools) | ✅ Good (7 tools, correct categories) | ✅ Most complete (8+ tools, wider source coverage) | Hallucinated multimeter & Allen keys at all k values |
| Safety warnings summary | ✅ 2 categories | ✅ 3 categories | ✅ Best — 3+ categories with electrical detail | Warnings scattered across corpus; all k levels miss some |
| Lubrication points | ⚠️ Partial (differential only) | ✅ Better (commutator added, 200-mile interval) | ✅ Same as k=5 (capped at 4 unique sources) | Many lubrication points not retrieved even at k=10 |

**Can the model successfully combine information from multiple chunks?**

Yes, for well-represented topics. k=10 for "maintenance tasks" correctly synthesized oil, water, gas mixture, spark plugs, and running gear checks from 6 different source files. For "safety warnings" and "tune-up tools," the model correctly aggregated across 5–7 unique sources.

**Does it miss information not retrieved?**

Consistently. The lubrication query hit a retrieval ceiling at k=5 (only 4 unique source files regardless of k increasing to 10). Information about front axle lubrication, wheel bearing grease, and individual oiling points was never retrieved because the embedding scores for those chunks were too low (~0.38) to surface above general maintenance content.

**Contradictory/overlapping chunks:**

The duplicate TXT+OCR document structure (each file exists as both .txt and -ocr.pdf) caused near-identical chunks from different source files to dominate top-k slots. At k=10 for maintenance tasks, 4 of 6 "unique" sources were txt/pdf duplicates of the same underlying content, effectively giving only 3 distinct sources.

**Optimal k for synthesis vs. targeted lookup:**

- **Targeted lookup** (what is the spark plug gap?): k=3–5 is sufficient; higher k adds noise.
- **Synthesis queries** (all maintenance tasks, all lubrication points): k=10–20 necessary; even then, low-scoring chunks with relevant content may be missed.
- **Core lesson:** RAG as implemented here is better suited for targeted retrieval than exhaustive synthesis. For truly comprehensive summaries, a full-document read (no chunking) or a dedicated summarization pass over the full corpus is more reliable.

In [12]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")


✓ Saved rag_index.faiss + .chunks
